In [1]:
import pickle
import numpy as np
import pandas as pd
import sklearn as sk
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error

In [2]:
URL1 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
URL2 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet"

In [3]:
def read_dataframe(url):
    df = pd.read_parquet(url)
    df.tpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)
    df.tpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)
    df['duration'] = (df.tpep_dropoff_datetime - df.tpep_pickup_datetime).dt.total_seconds() / 60
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df

## Q1. Downloading the data

In [4]:
df_train = read_dataframe(URL1)
len(df_train.columns)

20

## Q2. Computing duration

In [5]:
df_train['duration'].std()

np.float64(42.59435124195457)

## Q3. Dropping outliers

In [6]:
mask = (df_train['duration'] > 1) & (df_train['duration'] <= 60)
fraction = mask.mean() * 100 
fraction

np.float64(98.11286547457485)

In [7]:
df_train = df_train.loc[mask]
df_train['PU_DO'] = df_train['PULocationID'] + '_' +  df_train['DOLocationID']

In [8]:
df_val = read_dataframe(URL2)
df_val = df_val.loc[(df_val['duration'] > 1) & (df_val['duration'] <= 60)]
df_val['PU_DO'] = df_val['PULocationID'] + '_' +  df_val['DOLocationID']

## Q4. One-hot encoding

In [9]:
numerical = ['trip_distance']
categorical = ['PU_DO']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

X_train.indices = X_train.indices.astype(np.int32).astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32).astype(np.int32)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)
X_val, X_train

target = 'duration'
y_train = df_train[target]
y_val = df_val[target].values
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6017784 stored elements and shape (3008892, 21802)>

## Q5. Training a model

In [10]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)

root_mean_squared_error(y_train, y_pred)

5.14433946723173

## Q6. Evaluating the model

In [11]:
y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

5.2559583114075545